# IPython cell tracking

Create a workspace and save its key first. Run cells in order. Credentials use hidden prompts.


In [ ]:
%pip install -q "crashlens[jupyter,gpu] @ git+https://github.com/kavinsaravan/ML-Failure-Doctor.git#subdirectory=crashlens-sdk"

In [ ]:
import os, getpass, requests
from crashlens import WorkloadTracker
api_url = (os.getenv("CRASHLENS_URL") or input("Backend HTTPS URL: ")).strip().rstrip("/")
api_key = os.getenv("CRASHLENS_API_KEY") or getpass.getpass("Your saved CrashLens key: ")
fireworks_key = os.getenv("FIREWORKS_API_KEY") or getpass.getpass("Your Fireworks key (blank for rules): ")
fireworks_model = (os.getenv("FIREWORKS_MODEL") or input("Tool-capable Fireworks model: ")) if fireworks_key else None
tracker = WorkloadTracker(api_url, api_key=api_key, fireworks_api_key=fireworks_key or None, fireworks_model=fireworks_model)
response = requests.get(api_url + "/session", headers=tracker.headers, timeout=15)
response.raise_for_status()
print("Connected to your private workspace")

In [ ]:
os.environ["CRASHLENS_URL"] = api_url
os.environ["CRASHLENS_API_KEY"] = api_key
os.environ["FIREWORKS_API_KEY"] = fireworks_key or ""
os.environ["FIREWORKS_MODEL"] = fireworks_model or ""
%load_ext crashlens.jupyter
%crashlens_init $api_url

In [ ]:
%%crashlens_track "CPU calculation"
print(sum(range(100000)))

In [ ]:
%%crashlens_track "Missing dependency"
# This cell fails intentionally. Auto-diagnosis can use your Fireworks credits.
import crashlens_intentionally_missing_dependency_847263

In [ ]:
response = requests.get(api_url + "/workloads", headers=tracker.headers, timeout=15)
response.raise_for_status()
for workload in response.json():
    print(workload["id"], workload["name"], workload["status"])
print("Open your frontend dashboard using the same CrashLens key.")